In [ ]:
import pandas as pd
import re
import nltk
import time

# nltk.download('punkt_tab')
# nltk.download('stopwords', quiet=True)
# nltk.download('wordnet', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Bidirectional, LSTM
from sklearn.model_selection import train_test_split

In [ ]:
training_data = pd.read_csv("twitter_training.csv",header=None,names=["ID", "Topic", "Sentiment", "Tweet"])
training_data.drop(["ID", "Topic"], axis=1, inplace=True)
training_data.head(3)

,Sentiment,Tweet
0,Positive,im getting on borderlands and i will murder yo...
1,Positive,I am coming to the borders and I will kill you...
2,Positive,im getting on borderlands and i will kill you ...


In [ ]:
testing_data = pd.read_csv("twitter_validation.csv",header=None,names=["ID", "Topic", "Sentiment", "Tweet"])
testing_data.drop(["ID", "Topic"], axis=1, inplace=True)
testing_data.head(3)

,Sentiment,Tweet
0,Irrelevant,I mentioned on Facebook that I was struggling ...
1,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
2,Negative,@Microsoft Why do I pay for WORD when it funct...


#Check Nulls

In [ ]:
training_data.isnull().sum()

,0
Sentiment,0
Tweet,686


In [ ]:
training_data.dropna(inplace=True)
training_data.isnull().sum()

,0
Sentiment,0
Tweet,0


In [ ]:
testing_data.isnull().sum()

,0
Sentiment,0
Tweet,0


#Preprocessing

In [ ]:
training_data['Sentiment'] = training_data['Sentiment'].str.strip().str.lower()
training_data['Sentiment'] = training_data['Sentiment'].map({'negative': 0,'neutral': 1,'positive': 2,'irrelevant': 3})

testing_data['Sentiment'] = testing_data['Sentiment'].str.strip().str.lower()
testing_data['Sentiment'] = testing_data['Sentiment'].map({'negative': 0,'neutral': 1,'positive': 2,'irrelevant': 3})

In [ ]:
X_train = training_data["Tweet"]
y_train = training_data["Sentiment"]

X_test = testing_data["Tweet"]
y_test = testing_data["Sentiment"]

In [ ]:
X_train = X_train.astype(str)
X_test = X_test.astype(str)

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

In [ ]:
max_length = 20

X_train = pad_sequences(X_train, maxlen=max_length, padding="post")
X_test = pad_sequences(X_test, maxlen=max_length, padding="post")

print(X_train)

[[ 140    4    2 ...    0    0    0]
 [   3    1 6744 ...    0    0    0]
 [ 140    4    2 ...    0    0    0]
 ...
 [   6   14 2008 ...  118 1025 2157]
 [ 270  216  782 ...   80   47 2157]
 [   6   14 2008 ...    2 1025 2157]]


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

#Task 1: Build the Baseline Model

##RNNsimple Model

In [ ]:
vocab_size = len(tokenizer.word_index) + 1

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    )
)

model.add(SimpleRNN(32))

model.add(Dense(4, activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

##Task 1 - Record the following:
• Training Accuracy: 0.917

• Validation Accuracy: 0.748

• Test Accuracy: 0.847

• Training Time: 695.75s

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

end = time.time()

training_time = end - start

print(f"Training Time: {training_time:.2f} seconds")

Epoch 1/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 51s 28ms/step - accuracy: 0.9170 - loss: 0.1912 - val_accuracy: 0.7581 - val_loss: 0.7772
Epoch 2/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 82s 28ms/step - accuracy: 0.9168 - loss: 0.1877 - val_accuracy: 0.7811 - val_loss: 0.7130
Epoch 3/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 50s 27ms/step - accuracy: 0.9203 - loss: 0.1798 - val_accuracy: 0.7785 - val_loss: 0.7534
Epoch 4/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 82s 27ms/step - accuracy: 0.9201 - loss: 0.1782 - val_accuracy: 0.7797 - val_loss: 0.7507
Epoch 5/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 52s 28ms/step - accuracy: 0.9216 - loss: 0.1759 - val_accuracy: 0.7728 - val_loss: 0.7832
Epoch 6/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 81s 28ms/step - accuracy: 0.9212 - loss: 0.1751 - val_accuracy: 0.7684 - val_loss: 0.7750
Epoch 7/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 81s 27ms/step - accuracy: 0.9229 - loss: 0.1742 - val_accuracy: 0.7686 - val_loss: 0.8027
Epoch 8/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 50s 27ms/step - accuracy: 0.9238 -

In [ ]:
train_loss, train_accuracy = model.evaluate(
    X_train,
    y_train,
    verbose=0
)

print("Training Accuracy:", train_accuracy)

Training Accuracy: 0.9179674386978149


In [ ]:
val_loss, val_accuracy = model.evaluate(
    X_val,
    y_val,
    verbose=0
)

print("Validation Accuracy:", val_accuracy)

Validation Accuracy: 0.7487162351608276


In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.847000002861023


##Task 1 - Questions

1. What is the role of the Embedding layer?

The Embedding layer converts each word into a dense numerical vector of fixed size. it learns meaningful word embeddings during training.

2. Why do we use SimpleRNN instead of a Dense layer?

A SimpleRNN processes text sequentially and maintains information from previous words through its hidden state. This enables it to learn contextual and sequential relationships in text, whereas a Dense layer treats each input independently and cannot model word order or context.

3. What is the purpose of the Sigmoid activation function?

The sigmoid activation function maps outputs to values between 0 and 1, making it suitable for estimating probabilities in binary classification tasks.

#Task 2: Increase the Number of RNN Units

Modify the model by changing the number of units from 32 to 64

In [ ]:
vocab_size = len(tokenizer.word_index) + 1

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    )
)

model.add(SimpleRNN(64))

model.add(Dense(4, activation="softmax"))

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

##Task 2 - Record:

• Test Accuracy: 0.847

• Training Time: 610.12s



In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

end = time.time()

training_time = end - start

print(f"Training Time: {training_time:.2f} seconds")

Epoch 1/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 51s 28ms/step - accuracy: 0.6126 - loss: 0.9517 - val_accuracy: 0.7471 - val_loss: 0.6811
Epoch 2/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 83s 28ms/step - accuracy: 0.8428 - loss: 0.4140 - val_accuracy: 0.7893 - val_loss: 0.5778
Epoch 3/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 53s 29ms/step - accuracy: 0.8889 - loss: 0.2805 - val_accuracy: 0.7745 - val_loss: 0.6262
Epoch 4/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 81s 28ms/step - accuracy: 0.9024 - loss: 0.2371 - val_accuracy: 0.7868 - val_loss: 0.6594
Epoch 5/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 53s 28ms/step - accuracy: 0.9069 - loss: 0.2171 - val_accuracy: 0.7710 - val_loss: 0.6868
Epoch 6/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 52s 28ms/step - accuracy: 0.9127 - loss: 0.2074 - val_accuracy: 0.7718 - val_loss: 0.7081
Epoch 7/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 52s 28ms/step - accuracy: 0.9140 - loss: 0.2016 - val_accuracy: 0.7614 - val_loss: 0.7578
Epoch 8/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 52s 28ms/step - accuracy: 0.9156 -

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.8479999899864197


##Task 2 - Questions

1. Did the model performance improve? Yes, but only slightly

- 32 units: Test Accuracy = 84.70%
- 64 units: Test Accuracy = 84.80%



---



The validation accuracy also reached a slightly higher peak with 64 units (78.93%) compared to 32 units (78.11%)

2. Did the training time increase? No

The training time decreased slightly:

- 32 units: 695.75 seconds
- 64 units: 610.12 seconds

3. Explain your observations
- Both models achieved high training accuracy (about 91.8%), indicating that they learned the training data well.
- The test accuracy remained almost unchanged after increasing the number of units, suggesting that 32 RNN units were already sufficient to capture most of the useful patterns in the dataset.
- The validation accuracy peaked during the first few epochs and then gradually decreased, while the training accuracy continued to increase. This indicates that the model began to overfit the training data after several epochs.

#Task 3: Build a Stacked RNN

In [ ]:
model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    )
)

# First RNN layer
model.add(
    SimpleRNN(
        16,
        return_sequences=True
    )
)

# Second RNN layer
model.add(
    SimpleRNN(16)
)

# Output layer
model.add(
    Dense(4, activation="softmax")
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

##Task 3 - Record:
• Test Accuracy: 0.870

• Training Time: 616.7s

In [ ]:
start_time = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

end_time = time.time()

training_time = end_time - start_time

Epoch 1/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 66s 33ms/step - accuracy: 0.5654 - loss: 1.0372 - val_accuracy: 0.7076 - val_loss: 0.7677
Epoch 2/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 59s 32ms/step - accuracy: 0.8060 - loss: 0.5177 - val_accuracy: 0.7551 - val_loss: 0.6532
Epoch 3/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 58s 32ms/step - accuracy: 0.8671 - loss: 0.3411 - val_accuracy: 0.7574 - val_loss: 0.6595
Epoch 4/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 59s 32ms/step - accuracy: 0.8892 - loss: 0.2756 - val_accuracy: 0.7566 - val_loss: 0.6902
Epoch 5/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 58s 32ms/step - accuracy: 0.8977 - loss: 0.2483 - val_accuracy: 0.7582 - val_loss: 0.7325
Epoch 6/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 82s 31ms/step - accuracy: 0.9029 - loss: 0.2308 - val_accuracy: 0.7610 - val_loss: 0.7574
Epoch 7/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 59s 32ms/step - accuracy: 0.9063 - loss: 0.2217 - val_accuracy: 0.7580 - val_loss: 0.7831
Epoch 8/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 58s 31ms/step - accuracy: 0.9096 -

In [ ]:
print("Training Time:", training_time, "seconds")

Training Time: 616.7492291927338 seconds


In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.8700000047683716


##Task 3 - Questions

###1. Why is return_sequences=True required?

return_sequences=True is required in the first SimpleRNN layer because its output becomes the input to the second SimpleRNN layer.

###2. Did stacking RNN layers improve the results?

Yes.

Comparing the three models:
| Model                | Test Accuracy | Training Time |
| -------------------- | ------------: | ------------: |
| SimpleRNN (32 units) |     **84.7%** |      695.75 s |
| SimpleRNN (64 units) |     **84.8%** |      610.12 s |
| **Stacked RNN**      |     **87.0%** |      616.75 s |


The Stacked RNN achieved the highest test accuracy (87.0%), improving by approximately:

- 2.3% compared to the SimpleRNN with 32 units
- 2.2% compared to the SimpleRNN with 64 units

###3. Is adding more layers always beneficial? Explain.

No.

Adding more RNN layers can improve performance because deeper networks are able to learn more complex and hierarchical patterns in sequential data. This is evident in your experiment, where the Stacked RNN achieved the highest test accuracy.

However, adding more layers is not always beneficial because:
- It increases the model's complexity and training time.
- It requires more computational resources and memory.
- It increases the risk of overfitting, especially when the dataset is small or the model becomes too complex.
- After a certain depth, additional layers may provide little or no improvement and can even reduce generalization performance.

#Task 4: Build a Bidirectional RNN

In [ ]:
vocab_size = len(tokenizer.word_index) + 1

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    )
)

model.add(
    Bidirectional(
        SimpleRNN(64)
    )
)

model.add(Dense(4, activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

end = time.time()

training_time = end - start

Epoch 1/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 67s 34ms/step - accuracy: 0.6091 - loss: 0.9386 - val_accuracy: 0.7656 - val_loss: 0.6165
Epoch 2/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 80s 33ms/step - accuracy: 0.8526 - loss: 0.3815 - val_accuracy: 0.7820 - val_loss: 0.5830
Epoch 3/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 83s 33ms/step - accuracy: 0.8960 - loss: 0.2555 - val_accuracy: 0.7916 - val_loss: 0.5840
Epoch 4/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 60s 32ms/step - accuracy: 0.9090 - loss: 0.2188 - val_accuracy: 0.7924 - val_loss: 0.6296
Epoch 5/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 83s 33ms/step - accuracy: 0.9121 - loss: 0.2035 - val_accuracy: 0.7930 - val_loss: 0.6522
Epoch 6/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 62s 33ms/step - accuracy: 0.9163 - loss: 0.1929 - val_accuracy: 0.7769 - val_loss: 0.6925
Epoch 7/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 67s 36ms/step - accuracy: 0.9178 - loss: 0.1862 - val_accuracy: 0.7768 - val_loss: 0.7287
Epoch 8/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 70s 38ms/step - accuracy: 0.9202 -

##Task 4 - Record:
• Test Accuracy: 0.861

• Training Time: 693.77s

In [ ]:
print(f"Training Time: {training_time:.2f} seconds")

Training Time: 693.77 seconds


In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.8619999885559082


##Task 4 - Questions:

###1. Compare the results with the baseline model

The baseline model is the SimpleRNN with 64 units
| Model                   | Test Accuracy | Training Time |
| ----------------------- | ------------: | ------------: |
| Baseline (SimpleRNN 64) |     **84.8%** |  **610.12 s** |
| Bidirectional RNN       |     **86.2%** |  **693.77 s** |


- The Bidirectional RNN achieved a higher test accuracy (86.2%) than the baseline SimpleRNN (84.8%), giving an improvement of approximately 1.4%.
- The Bidirectional RNN also achieved a higher peak validation accuracy (79.3%) compared to the baseline (78.9%).
- However, the training time increased from 610.12 s to 693.77 s, since the Bidirectional RNN processes the sequence in both forward and backward directions, requiring more computations.

###2. Why might Bidirectional RNN achieve better performance?

A Bidirectional RNN processes the input sequence in both forward and backward directions. This enables the model to learn contextual information from both the preceding and following words.

###3. Is Bidirectional RNN suitable for real-time next-word prediction? Why or why not?

No

A Bidirectional RNN is not suitable for real-time next-word prediction because it requires access to the entire input sequence, including future words. During next-word prediction, future words are not yet available, so the backward RNN cannot be used.

#Task 5: Replace SimpleRNN with LSTM

In [ ]:
vocab_size = len(tokenizer.word_index) + 1

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    )
)

model.add(LSTM(64))

model.add(Dense(4, activation="softmax"))

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

##Task 5 - Record:
• Test Accuracy: 0.89

• Training Time: 736.30s

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

end = time.time()

training_time = end - start

print(f"Training Time: {training_time:.2f} seconds")

Epoch 1/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 81s 42ms/step - accuracy: 0.5963 - loss: 0.9709 - val_accuracy: 0.7026 - val_loss: 0.7353
Epoch 2/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 78s 40ms/step - accuracy: 0.8050 - loss: 0.5067 - val_accuracy: 0.8000 - val_loss: 0.5184
Epoch 3/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 74s 36ms/step - accuracy: 0.8549 - loss: 0.3597 - val_accuracy: 0.8054 - val_loss: 0.4881
Epoch 4/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 80s 35ms/step - accuracy: 0.8742 - loss: 0.3027 - val_accuracy: 0.8192 - val_loss: 0.5041
Epoch 5/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 67s 36ms/step - accuracy: 0.8853 - loss: 0.2699 - val_accuracy: 0.8232 - val_loss: 0.5175
Epoch 6/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 66s 36ms/step - accuracy: 0.8945 - loss: 0.2468 - val_accuracy: 0.8237 - val_loss: 0.5292
Epoch 7/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 80s 35ms/step - accuracy: 0.8993 - loss: 0.2307 - val_accuracy: 0.8283 - val_loss: 0.5458
Epoch 8/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 64s 35ms/step - accuracy: 0.9031 -

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.8999999761581421


##Task 5 - Questions:

###1. Compare the performance with SimpleRNN

| Model                | Test Accuracy | Training Time |
| -------------------- | ------------: | ------------: |
| SimpleRNN (32 units) |    **84.70%** |  **695.75 s** |
| SimpleRNN (64 units) |    **84.80%** |  **610.12 s** |
| LSTM (64 units)      |    **90.00%** |  **736.30 s** |

The LSTM outperformed the SimpleRNN. While the best SimpleRNN achieved a test accuracy of approximately 84.8%, the LSTM achieved 90.0%, an improvement of about 5.2 percentage points.

###2. Which model required more training time?

The LSTM required more training time.

###3. Why is LSTM generally better for long text sequences?

LSTM is generally better for long text sequences because it is designed to retain important information over long distances while discarding irrelevant information using its gating mechanism.

#Task 6: Build a Bidirectional LSTM

In [ ]:
vocab_size = len(tokenizer.word_index) + 1

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    )
)

model.add(Bidirectional(LSTM(64)))

model.add(Dense(4, activation="softmax"))

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

##Task 6 - Record:
• Test Accuracy: 0.88

• Training Time: 1085.99s

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

end = time.time()

training_time = end - start

print(f"Training Time: {training_time:.2f} seconds")

Epoch 1/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 103s 53ms/step - accuracy: 0.6109 - loss: 0.9353 - val_accuracy: 0.7470 - val_loss: 0.6308
Epoch 2/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 132s 48ms/step - accuracy: 0.8208 - loss: 0.4545 - val_accuracy: 0.8091 - val_loss: 0.5007
Epoch 3/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 88s 47ms/step - accuracy: 0.8654 - loss: 0.3266 - val_accuracy: 0.8243 - val_loss: 0.4673
Epoch 4/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 85s 46ms/step - accuracy: 0.8864 - loss: 0.2717 - val_accuracy: 0.8261 - val_loss: 0.4900
Epoch 5/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 83s 45ms/step - accuracy: 0.8974 - loss: 0.2408 - val_accuracy: 0.8344 - val_loss: 0.5015
Epoch 6/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 83s 45ms/step - accuracy: 0.9044 - loss: 0.2213 - val_accuracy: 0.8336 - val_loss: 0.5136
Epoch 7/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 146s 47ms/step - accuracy: 0.9100 - loss: 0.2044 - val_accuracy: 0.8359 - val_loss: 0.5471
Epoch 8/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 138s 45ms/step - accuracy: 0.91

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.8889999985694885


##Task 6 - Questions:

| Model              | Test Accuracy | Training Time (s) |
| ------------------ | ------------: | ----------------: |
| SimpleRNN (32)     |        84.70% |            695.75 |
| SimpleRNN (64)     |        84.80% |            610.12 |
| Stacked RNN        |        87.00% |            616.75 |
| Bidirectional RNN  |        86.20% |            693.77 |
| **LSTM**           |    **90.00%** |        **736.30** |
| Bidirectional LSTM |        88.90% |           1085.99 |


###1. Which model achieved the highest accuracy?

The LSTM model achieved the highest test accuracy of 90.0%. Although the Bidirectional LSTM is a more advanced architecture, it achieved 88.9%, which is slightly lower than the standard LSTM on this dataset.

###2. Which model required the longest training time?

The Bidirectional LSTM required the longest training time

###3. Was the performance improvement worth the additional complexity?

no

Although the Bidirectional LSTM is a more sophisticated model, it did not outperform the standard LSTM. In fact:

- LSTM: 90.0% test accuracy
- Bidirectional LSTM: 88.9% test accuracy

At the same time, the Bidirectional LSTM required about 350 more seconds of training (1085.99 s vs. 736.30 s).

#Task 7: Hyperparameter Experiment

In [ ]:
vocab_size = len(tokenizer.word_index) + 1

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    )
)

model.add(LSTM(128))

model.add(Dense(4, activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

##Task 7 - Record:
• Test Accuracy:

• Training Time:

In [ ]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

end = time.time()

training_time = end - start

print(f"Training Time: {training_time:.2f} seconds")

Epoch 1/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 108s 55ms/step - accuracy: 0.5735 - loss: 1.0063 - val_accuracy: 0.7289 - val_loss: 0.7117
Epoch 2/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 101s 54ms/step - accuracy: 0.8040 - loss: 0.5037 - val_accuracy: 0.7994 - val_loss: 0.5250
Epoch 3/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 104s 56ms/step - accuracy: 0.8561 - loss: 0.3564 - val_accuracy: 0.8164 - val_loss: 0.4796
Epoch 4/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 102s 55ms/step - accuracy: 0.8751 - loss: 0.2979 - val_accuracy: 0.8235 - val_loss: 0.4807
Epoch 5/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 100s 54ms/step - accuracy: 0.8877 - loss: 0.2643 - val_accuracy: 0.8186 - val_loss: 0.5041
Epoch 6/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 99s 53ms/step - accuracy: 0.8945 - loss: 0.2451 - val_accuracy: 0.8320 - val_loss: 0.4923
Epoch 7/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 142s 53ms/step - accuracy: 0.9014 - loss: 0.2276 - val_accuracy: 0.8249 - val_loss: 0.5177
Epoch 8/10
1850/1850 ━━━━━━━━━━━━━━━━━━━━ 98s 53ms/step - accuracy: 0.

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.8730000257492065


##Observation

Increasing the number of LSTM units made the model more complex and significantly increased the training time. However, instead of improving performance, the test accuracy decreased from 90.0% to 87.3%. This suggests that the larger model likely overfit the training data, learning dataset-specific patterns rather than generalizing well to unseen tweets. Therefore, for this dataset, 64 LSTM units provide a better balance between accuracy, training time, and model complexity.